# Module 1: Language Detection (LOCAL TRAINING)
Dataset pulled via HuggingFace `datasets` API (no manual download needed).
Runs entirely on CPU -- no GPU required for this module.

Approach: TF-IDF (char n-grams) + Logistic Regression
WHY: language ID relies on character/subword statistics, not deep semantics --
a lightweight classical model is fast to train and fast at inference (this runs
on every message, first in the pipeline).

In [1]:
# If not already installed in your local venv:
# pip install datasets scikit-learn joblib

import numpy as np
import pandas as pd
import joblib
import json
import os
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

## Load dataset (streamed from HuggingFace Hub, cached locally after first run)

In [2]:
ds = load_dataset("papluca/language-identification")
print(ds)

train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()
print(train_df.shape, val_df.shape, test_df.shape)

README.md:   0%|          | 0.00/4.99k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 12.0MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
})
(70000, 2) (10000, 2) (10000, 2)


## Preprocessing

In [3]:
def clean_text(s: str) -> str:
    return str(s).strip().lower()

for df in (train_df, val_df, test_df):
    df["text_clean"] = df["text"].apply(clean_text)

## Encode labels

In [4]:
le = LabelEncoder()
y_train = le.fit_transform(train_df["labels"])
y_val = le.transform(val_df["labels"])
y_test = le.transform(test_df["labels"])
label_names = list(le.classes_)
print(f"{len(label_names)} languages: {label_names}")

20 languages: ['ar', 'bg', 'de', 'el', 'en', 'es', 'fr', 'hi', 'it', 'ja', 'nl', 'pl', 'pt', 'ru', 'sw', 'th', 'tr', 'ur', 'vi', 'zh']


## Vectorize + Train

In [5]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 5), max_features=50000, sublinear_tf=True,
)
X_train = vectorizer.fit_transform(train_df["text_clean"])
X_val = vectorizer.transform(val_df["text_clean"])
X_test = vectorizer.transform(test_df["text_clean"])

clf = LogisticRegression(max_iter=1000, C=10, n_jobs=-1)  # n_jobs=-1 uses all 9 cores
clf.fit(X_train, y_train)

/home/alihamdi/Downloads/ecommerce-chatbot-local/local_app/venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",10
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"n_jobs n_jobs: int, default=NoneDoes not have any effect... deprecated:: 1.8 `n_jobs` is deprecated in version 1.8 and will be removed in 1.10.",-1
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None


## Evaluate

In [6]:
val_preds = clf.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_preds))

test_preds = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds, target_names=label_names))

Validation accuracy: 0.9952
Test accuracy: 0.9955
              precision    recall  f1-score   support

          ar       1.00      1.00      1.00       500
          bg       1.00      1.00      1.00       500
          de       1.00      1.00      1.00       500
          el       1.00      1.00      1.00       500
          en       1.00      1.00      1.00       500
          es       0.99      1.00      1.00       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.97      0.98       500
          it       1.00      1.00      1.00       500
          ja       1.00      1.00      1.00       500
          nl       1.00      1.00      1.00       500
          pl       1.00      1.00      1.00       500
          pt       1.00      0.99      1.00       500
          ru       1.00      1.00      1.00       500
          sw       0.94      1.00      0.97       500
          th       1.00      1.00      1.00       500
          tr       1.00      1.

## Save artifacts directly into the local_app models folder
Adjust OUTPUT_DIR if your folder layout differs.

In [7]:
OUTPUT_DIR = "../local_app/models/language"
os.makedirs(OUTPUT_DIR, exist_ok=True)

joblib.dump(vectorizer, os.path.join(OUTPUT_DIR, "lang_vectorizer.pkl"))
joblib.dump(clf, os.path.join(OUTPUT_DIR, "lang_model.pkl"))
with open(os.path.join(OUTPUT_DIR, "lang_labels.json"), "w") as f:
    json.dump({str(i): lbl for i, lbl in enumerate(label_names)}, f, indent=2)

print("Saved to", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Saved to ../local_app/models/language
['.gitkeep', 'lang_vectorizer.pkl', 'lang_model.pkl', 'lang_labels.json']
